# Lab 7: Healthcare Operations and Resource Optimization

**AI in Healthcare · Unit II · Student notebook · Approximately 120 minutes**  
**Companion lecture:** [Healthcare Operations and Resource Optimization](../lectures/session-07-healthcare-operations-resource-optimization.ipynb)

## Learning objectives

By the end of this lab, you should be able to:

1. Audit synthetic appointment requests and distinguish predictions from realized outcomes.
2. Trace a multi-clinician outpatient queue.
3. Calculate access, waiting, utilization, overtime, and urgent-delay measures.
4. Compare booking policies across repeated uncertain clinic days.
5. Test sensitivity to changing demand and model miscalibration.
6. Recommend a policy while stating safety, equity, and deployment limitations.

**Prerequisites:** Python, pandas, NumPy, descriptive statistics, and the companion lecture.  
**Dependencies:** Python 3, NumPy, pandas, and matplotlib. No downloads or installation cells are required.

All patient identifiers, probabilities, groups, events, and results in this notebook are deterministic synthetic teaching fixtures. The lab does not provide clinical advice or evidence that a policy is suitable for a real clinic.

## Scenario and decision frame

You support a fictional outpatient clinic with two clinicians and an eight-hour day. The normal template contains 30 appointment places. The manager is considering:

- retaining the baseline template;
- reserving places for uncertain same-day urgent demand;
- adding limited bookings to slots whose scheduled patient has a high predicted non-attendance probability; or
- using non-preemptive urgency priority when selecting the next waiting patient.

The schedule must be created **before the clinic opens**. Predicted non-attendance probability is available then. Actual attendance, arrival delay, consultation duration, and urgent requests are future realizations and must not be used to construct the schedule.

| Item | Definition |
|---|---|
| Intended user | Clinic manager |
| Unit in `requests` | One appointment request |
| Unit in a simulated queue | One attended visit or urgent request |
| Decision time | Before clinic opening |
| Action | Choose reservation, overbooking, and queue rules |
| Main outcomes | Access, waiting, overtime, utilization, urgent delay, subgroup effects |

The clinic and its 30-minute urgent-wait review threshold are illustrative, not clinical standards.

## Deliverables and rubric

Submit the completed notebook with:

1. data audit and interpretation;
2. validated baseline queue trace;
3. policy-comparison table over repeated days;
4. at least two plots showing both typical and high-delay outcomes;
5. demand and calibration sensitivity results;
6. subgroup audit with sample sizes; and
7. a concise recommendation memo with assumptions and limitations.

| Criterion | Marks |
|---|---:|
| Data audit, row meaning, and decision-time reasoning | 15 |
| Queue trace and operational metrics | 20 |
| Reproducible repeated-policy comparison | 25 |
| Uncertainty and sensitivity analysis | 15 |
| Safety and subgroup evaluation | 15 |
| Interpretation, recommendation, and limitations | 10 |
| **Total** | **100** |

Code that runs is necessary but not sufficient. Marks depend on explaining what the outputs mean and what the simulation cannot establish.

## Setup and generate the fixture — 10 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

CLINIC_CLOSE_MINUTE = 480  # 17:00 when minute 0 represents 09:00
BASE_TEMPLATE_SIZE = 30
N_CLINICIANS = 2
URGENT_WAIT_REVIEW_MIN = 30  # illustrative lab rule, not a clinical standard

In [ ]:
def make_synthetic_requests(seed=7, n_requests=40):
    # Create fictional requests and scores available before the clinic day.
    rng = np.random.default_rng(seed)
    access_group = rng.choice(
        ["near_clinic", "far_from_clinic"],
        size=n_requests,
        p=[0.65, 0.35],
    )
    travel_minutes = np.where(
        access_group == "near_clinic",
        rng.integers(5, 31, n_requests),
        rng.integers(35, 91, n_requests),
    )
    prior_missed = rng.binomial(2, np.where(access_group == "near_clinic", 0.12, 0.22))
    raw_score = -2.8 + 0.018 * travel_minutes + 0.75 * prior_missed
    predicted_no_show = 1 / (1 + np.exp(-raw_score))

    requests = pd.DataFrame(
        {
            "request_id": [f"R{i:03d}" for i in range(n_requests)],
            "patient_id": [f"P{i:03d}" for i in range(n_requests)],
            "request_order": np.arange(1, n_requests + 1),
            "access_group": access_group,
            "travel_minutes": travel_minutes,
            "prior_missed_appointments": prior_missed,
            "predicted_no_show_probability": np.clip(predicted_no_show, 0.05, 0.60),
        }
    )
    return requests


requests = make_synthetic_requests()
requests.head()

The fixture is **hand-created synthetic data**, not an export from a clinical system. `access_group` exists to investigate whether a policy produces different outcomes; it is not a clinical urgency label.

The no-show score is generated partly from travel time and prior missed appointments. This deliberately creates an ethical issue: a scheduling rule may convert an access barrier into worse service for the people affected by it.

## Exercise 1: Audit appointment requests — 10 minutes

1. Confirm the row count and uniqueness of `request_id` and `patient_id`.
2. Check ranges and missingness.
3. Summarize request count and predicted non-attendance probability by `access_group`.
4. State whether `predicted_no_show_probability` is a prediction, decision, or observed outcome.
5. Explain why the eventual `showed_up` value must not appear in this table.

Complete the cell below. Preserve one row per appointment request.

In [ ]:
# TODO: audit `requests` and create a subgroup summary named `request_audit_by_group`.
# Suggested operations: len, nunique, isna, groupby, agg, min, and max.

request_audit_by_group = None
request_audit_by_group

In [ ]:
assert requests["request_id"].is_unique, "request_id must be unique"
assert requests["patient_id"].is_unique, "This fixture expects one request per patient"
assert requests["predicted_no_show_probability"].between(0, 1).all()

if request_audit_by_group is None:
    print("Complete Exercise 1: create request_audit_by_group.")
else:
    display(request_audit_by_group)

## Build schedules and simulate the queue

The following functions are provided infrastructure. Read their docstrings before using them.

`build_schedule` creates a decision using only pre-opening information. Reserving a place reduces routine bookings. Score-guided overbooking assigns additional requests to times associated with the highest predicted non-attendance probabilities; it does not deny the original patient an appointment.

In [ ]:
def build_schedule(
    request_table,
    reserve_slots=0,
    overbook_slots=0,
    score_guided_overbooking=False,
    base_template_size=BASE_TEMPLATE_SIZE,
):
    # Create a pre-opening schedule without using realized attendance.
    if reserve_slots < 0 or overbook_slots < 0:
        raise ValueError("reserve_slots and overbook_slots must be non-negative")
    regular_places = base_template_size - reserve_slots
    if regular_places <= 0:
        raise ValueError("reserve_slots leave no routine appointment places")
    if regular_places + overbook_slots > len(request_table):
        raise ValueError("Not enough appointment requests for this policy")

    template = np.repeat(np.arange(0, 450, 30), 2)
    if reserve_slots:
        # Spread nominal reserved capacity across the day instead of simply
        # removing the final appointments. Same-day arrivals still join the queue.
        reserved_positions = np.floor(
            (np.arange(reserve_slots) + 0.5) * len(template) / reserve_slots
        ).astype(int)
        regular_template = np.delete(template, reserved_positions)
    else:
        regular_template = template
    selected = request_table.sort_values("request_order").head(regular_places).copy()
    selected["scheduled_minute"] = regular_template[:regular_places]
    selected["booking_type"] = "standard"

    if overbook_slots:
        waitlist = (
            request_table.sort_values("request_order")
            .iloc[regular_places : regular_places + overbook_slots]
            .copy()
        )
        if score_guided_overbooking:
            host_slots = selected.nlargest(
                overbook_slots, "predicted_no_show_probability"
            )["scheduled_minute"].to_numpy()
        else:
            host_slots = selected.tail(overbook_slots)["scheduled_minute"].to_numpy()
        waitlist["scheduled_minute"] = host_slots
        waitlist["booking_type"] = "overbooked"
        selected = pd.concat([selected, waitlist], ignore_index=True)

    return selected.sort_values(
        ["scheduled_minute", "booking_type", "request_order"], kind="stable"
    ).reset_index(drop=True)


baseline_schedule = build_schedule(requests)
baseline_schedule[[
    "request_id",
    "patient_id",
    "scheduled_minute",
    "booking_type",
    "predicted_no_show_probability",
]].head(8)

In [ ]:
def process_queue(arrivals, n_clinicians=N_CLINICIANS, queue_rule="fcfs"):
    # Run a non-preemptive multi-clinician queue. FCFS selects the
    # longest-waiting arrival; urgent_priority selects urgent visits first,
    # then by arrival. A consultation in progress is never interrupted.
    if queue_rule not in {"fcfs", "urgent_priority"}:
        raise ValueError("queue_rule must be 'fcfs' or 'urgent_priority'")
    if n_clinicians < 1:
        raise ValueError("n_clinicians must be at least 1")

    pending = arrivals.sort_values("arrival_minute", kind="stable").reset_index(drop=True)
    server_available = np.zeros(n_clinicians, dtype=float)
    waiting = []
    records = []
    next_arrival_index = 0
    current_time = 0.0

    while next_arrival_index < len(pending) or waiting:
        if not waiting:
            current_time = max(
                current_time,
                float(pending.loc[next_arrival_index, "arrival_minute"]),
            )

        while (
            next_arrival_index < len(pending)
            and pending.loc[next_arrival_index, "arrival_minute"] <= current_time
        ):
            waiting.append(pending.loc[next_arrival_index].to_dict())
            next_arrival_index += 1

        free_servers = np.flatnonzero(server_available <= current_time)
        while waiting and len(free_servers):
            if queue_rule == "urgent_priority":
                chosen_index = min(
                    range(len(waiting)),
                    key=lambda i: (
                        0 if waiting[i]["urgency"] == "urgent" else 1,
                        waiting[i]["arrival_minute"],
                        waiting[i]["visit_id"],
                    ),
                )
            else:
                chosen_index = min(
                    range(len(waiting)),
                    key=lambda i: (waiting[i]["arrival_minute"], waiting[i]["visit_id"]),
                )

            visit = waiting.pop(chosen_index)
            server = int(free_servers[0])
            start = current_time
            finish = start + float(visit["service_duration_min"])
            records.append(
                {
                    **visit,
                    "clinician": server + 1,
                    "service_start_minute": start,
                    "completion_minute": finish,
                    "wait_min": start - float(visit["arrival_minute"]),
                }
            )
            server_available[server] = finish
            free_servers = np.flatnonzero(server_available <= current_time)

        if waiting:
            current_time = float(server_available.min())
        elif next_arrival_index < len(pending):
            current_time = max(
                current_time,
                float(pending.loc[next_arrival_index, "arrival_minute"]),
            )

    return pd.DataFrame(records).sort_values(
        ["service_start_minute", "clinician"], kind="stable"
    ).reset_index(drop=True)


def simulate_clinic_day(
    schedule,
    seed,
    n_clinicians=N_CLINICIANS,
    queue_rule="fcfs",
    urgent_rate=4.0,
    no_show_logit_shift=0.0,
):
    # Realize attendance and demand, then process one fictional clinic day.
    # no_show_logit_shift changes actual non-attendance odds while leaving the
    # pre-opening predicted probabilities unchanged. It supports a drift test.
    # Independent streams keep urgent demand and shared patient outcomes aligned
    # when the same seed is used to compare policies of different sizes.
    streams = np.random.SeedSequence(seed).spawn(6)
    attendance_rng, arrival_rng, service_rng, urgent_count_rng, urgent_arrival_rng, urgent_service_rng = [
        np.random.default_rng(stream) for stream in streams
    ]
    regular = schedule.copy()
    predicted = regular["predicted_no_show_probability"].to_numpy()
    predicted_logit = np.log(predicted / (1 - predicted))
    actual_no_show_probability = 1 / (
        1 + np.exp(-(predicted_logit + no_show_logit_shift))
    )
    showed_up = attendance_rng.random(len(regular)) >= actual_no_show_probability

    attended = regular.loc[showed_up].copy()
    attended["visit_id"] = "V-" + attended["request_id"]
    attended["source"] = "scheduled"
    attended["urgency"] = "routine"
    attended["arrival_minute"] = np.maximum(
        0,
        attended["scheduled_minute"].to_numpy()
        + arrival_rng.normal(0, 8, len(attended)),
    )
    attended["service_duration_min"] = np.clip(
        service_rng.lognormal(np.log(24), 0.35, len(attended)), 8, 60
    )

    urgent_count = urgent_count_rng.poisson(urgent_rate)
    urgent = pd.DataFrame(
        {
            "visit_id": [f"U-{seed}-{i}" for i in range(urgent_count)],
            "request_id": pd.NA,
            "patient_id": [f"UP-{seed}-{i}" for i in range(urgent_count)],
            "request_order": pd.NA,
            "access_group": "not_recorded_for_urgent_fixture",
            "travel_minutes": np.nan,
            "prior_missed_appointments": np.nan,
            "predicted_no_show_probability": np.nan,
            "scheduled_minute": np.nan,
            "booking_type": "same_day_request",
            "source": "same_day",
            "urgency": "urgent",
            "arrival_minute": urgent_arrival_rng.uniform(0, 420, urgent_count),
            "service_duration_min": np.clip(
                urgent_service_rng.lognormal(np.log(30), 0.35, urgent_count), 10, 75
            ),
        }
    )

    arrivals = pd.concat([attended, urgent], ignore_index=True, sort=False)
    return process_queue(arrivals, n_clinicians=n_clinicians, queue_rule=queue_rule)

### Validate the queue engine

Before trusting a complex simulation, test a deterministic case whose answer can be calculated by hand. The assertions below expect waits of 0, 15, and 10 minutes for one clinician.

In [ ]:
toy_arrivals = pd.DataFrame(
    {
        "visit_id": ["P1", "P2", "P3"],
        "patient_id": ["P1", "P2", "P3"],
        "source": ["scheduled"] * 3,
        "urgency": ["routine"] * 3,
        "access_group": ["toy"] * 3,
        "arrival_minute": [0.0, 5.0, 25.0],
        "service_duration_min": [20.0, 15.0, 10.0],
    }
)
toy_result = process_queue(toy_arrivals, n_clinicians=1, queue_rule="fcfs")

assert toy_result["wait_min"].tolist() == [0.0, 15.0, 10.0]
assert toy_result["completion_minute"].tolist() == [20.0, 35.0, 45.0]
print("Deterministic FCFS check passed.")
toy_result[[
    "visit_id", "arrival_minute", "service_start_minute",
    "completion_minute", "wait_min"
]]

## Exercise 2: Trace one baseline clinic day — 15 minutes

1. Simulate seed `2026` using `baseline_schedule` and FCFS.
2. Check that waits and service durations are non-negative.
3. Display the five longest waits.
4. Explain why `len(day)` is the number of attended/same-day visits, not the number of scheduled appointments or unique requests.
5. Plot a clinician timeline using horizontal bars. Use different colours for routine and urgent visits.

In [ ]:
baseline_day = simulate_clinic_day(
    baseline_schedule,
    seed=2026,
    n_clinicians=N_CLINICIANS,
    queue_rule="fcfs",
)

assert (baseline_day["wait_min"] >= 0).all()
assert (baseline_day["service_duration_min"] > 0).all()

# TODO: display the five longest waits and create a clinician timeline.
baseline_day.head()

## Exercise 3: Calculate operational outcomes — 20 minutes

Implement `summarise_day`. Return a dictionary containing at least:

- `visits`
- `started_by_close`
- `mean_wait_min`
- `median_wait_min`
- `p90_wait_min`
- `max_wait_min`
- `utilization_before_close`
- `overtime_min`
- `urgent_visits`
- `urgent_wait_breaches`

For utilization, count only the part of each consultation falling between minute 0 and minute 480:

$$
\text{busy before close}=
\sum_i \max(0,\min(f_i,480)-\max(s_i,0)),
$$

then divide by `n_clinicians × 480`. Here $s_i$ and $f_i$ are visit $i$'s service start and finish.

In [ ]:
def summarise_day(
    visits,
    n_clinicians=N_CLINICIANS,
    closing_minute=CLINIC_CLOSE_MINUTE,
    urgent_wait_review_min=URGENT_WAIT_REVIEW_MIN,
):
    # TODO: calculate the requested metrics and return them as a dictionary.
    # Hint for p90: visits["wait_min"].quantile(0.90)
    # Hint for busy time: use np.minimum, np.maximum, and np.clip.
    return None


baseline_metrics = summarise_day(baseline_day)
if baseline_metrics is None:
    print("Complete summarise_day to see the baseline metrics.")
else:
    display(pd.Series(baseline_metrics, name="baseline_seed_2026"))

### Metric checks

Use checks as debugging aids, not as substitutes for interpretation. The utilization definition above should remain between 0 and 1 because it counts only busy time inside the staffed window. Overtime is reported separately.

In [ ]:
if baseline_metrics is not None:
    required_metrics = {
        "visits", "started_by_close", "mean_wait_min", "median_wait_min",
        "p90_wait_min", "max_wait_min", "utilization_before_close",
        "overtime_min", "urgent_visits", "urgent_wait_breaches",
    }
    assert required_metrics.issubset(baseline_metrics)
    assert 0 <= baseline_metrics["utilization_before_close"] <= 1
    assert baseline_metrics["mean_wait_min"] <= baseline_metrics["max_wait_min"]
    assert baseline_metrics["urgent_wait_breaches"] <= baseline_metrics["urgent_visits"]
    print("Metric checks passed.")
else:
    print("Metric checks skipped until summarise_day is completed.")

## Exercise 4: Compare scheduling policies — 25 minutes

The candidate policies below change one or more decisions:

- **Baseline:** 30 standard bookings, FCFS.
- **Reserve two:** 28 standard bookings, leaving nominal capacity for same-day demand.
- **Score-guided overbook two:** 30 standard plus two additional requests assigned to high predicted no-show slots.
- **Urgent priority:** baseline bookings with non-preemptive priority for urgent arrivals.

First inspect schedule counts and duplicate slot times. Then run every policy for at least 200 days. Use the same simulation seeds for each policy so that comparisons are not driven merely by different random-number sequences.

In [ ]:
policy_definitions = {
    "baseline_fcfs": {
        "schedule": build_schedule(requests),
        "queue_rule": "fcfs",
    },
    "reserve_2_fcfs": {
        "schedule": build_schedule(requests, reserve_slots=2),
        "queue_rule": "fcfs",
    },
    "overbook_2_score_guided": {
        "schedule": build_schedule(
            requests,
            overbook_slots=2,
            score_guided_overbooking=True,
        ),
        "queue_rule": "fcfs",
    },
    "baseline_urgent_priority": {
        "schedule": build_schedule(requests),
        "queue_rule": "urgent_priority",
    },
}

pd.DataFrame(
    [
        {
            "policy": name,
            "scheduled_requests": len(spec["schedule"]),
            "overbooked_requests": (spec["schedule"]["booking_type"] == "overbooked").sum(),
            "queue_rule": spec["queue_rule"],
        }
        for name, spec in policy_definitions.items()
    ]
)

In [ ]:
def evaluate_policy(
    policy_name,
    policy_spec,
    seeds,
    urgent_rate=4.0,
    no_show_logit_shift=0.0,
):
    # Run one policy repeatedly after summarise_day has been implemented.
    rows = []
    for simulation_id, seed in enumerate(seeds):
        visits = simulate_clinic_day(
            policy_spec["schedule"],
            seed=int(seed),
            n_clinicians=N_CLINICIANS,
            queue_rule=policy_spec["queue_rule"],
            urgent_rate=urgent_rate,
            no_show_logit_shift=no_show_logit_shift,
        )
        metrics = summarise_day(visits)
        if metrics is None:
            return None
        rows.append(
            {
                "policy": policy_name,
                "simulation_id": simulation_id,
                "seed": int(seed),
                **metrics,
            }
        )
    return pd.DataFrame(rows)


simulation_seeds = np.arange(20_000, 20_300)

# TODO: call evaluate_policy for every policy, concatenate the returned frames,
# and store the result in `policy_results`.
policy_results = None

if policy_results is None:
    print("Complete Exercise 4 to create policy_results.")
else:
    display(policy_results.head())

Create a policy summary with:

- median visits and started-by-close count;
- median mean wait;
- median and 90th percentile of daily `p90_wait_min`;
- median utilization;
- 90th-percentile overtime; and
- mean urgent-wait breaches.

Then plot at least `mean_wait_min` and `overtime_min` distributions by policy. Explain why reporting only the mean across all simulations would be insufficient.

In [ ]:
# TODO: create `policy_summary` and at least two policy-comparison plots.
policy_summary = None

if policy_summary is None:
    print("Create policy_summary after policy_results is available.")
else:
    display(policy_summary)

### Interpretation prompts

1. Which policy completes or starts the most visits before closing?
2. Which policy has the lowest tail waiting and overtime?
3. Does urgent priority improve urgent-delay outcomes? What happens to routine waiting?
4. Does any policy dominate every other policy, or are there trade-offs?
5. Which measures would you treat as constraints rather than components of a weighted score?

## Exercise 5: Stress-test assumptions — 15 minutes

Policies should not be selected using only one assumed future. Repeat the policy comparison under:

1. **Demand surge:** set `urgent_rate=7.0`.
2. **Attendance drift:** set `no_show_logit_shift=-0.6`. This makes actual non-attendance less likely than the unchanged predictions suggest.

Store the two summaries as `surge_summary` and `attendance_drift_summary`. Identify any policy whose apparent advantage disappears or whose safety/workload outcomes worsen materially.

Explain why this is a policy stress test rather than external clinical validation.

In [ ]:
# TODO: repeat evaluation with urgent_rate=7.0.
surge_results = None
surge_summary = None

# TODO: repeat evaluation with no_show_logit_shift=-0.6.
attendance_drift_results = None
attendance_drift_summary = None

print("Demand-surge summary ready:", surge_summary is not None)
print("Attendance-drift summary ready:", attendance_drift_summary is not None)

## Exercise 6: Audit subgroup effects — 15 minutes

The score partly reflects travel time, so score-guided overbooking may affect the two synthetic access groups differently.

For the score-guided policy:

1. Count standard and overbooked requests by `access_group`.
2. Across at least 200 simulated days, calculate attended sample size, mean wait, and 90th-percentile wait by group.
3. Compare those results with the baseline.
4. Report sample sizes beside every subgroup measure.
5. Explain why an observed difference is a reason to investigate, not proof of discrimination or harm.

Do not treat missing `access_group` for synthetic urgent arrivals as one of the scheduled-patient groups.

In [ ]:
# TODO: create a booking audit and repeated-day subgroup results.
booking_audit_by_group = None
subgroup_results = None

if booking_audit_by_group is None:
    print("Create booking_audit_by_group.")
else:
    display(booking_audit_by_group)

if subgroup_results is None:
    print("Create subgroup_results with sample sizes and waiting measures.")
else:
    display(subgroup_results)

## Exercise 7: Recommendation memo — 10 minutes

Write 150–250 words addressed to the fictional clinic manager. Include:

1. the policy you recommend for a limited pilot—or why none should be piloted;
2. evidence from access, tail waiting, overtime, and urgent-delay outcomes;
3. what happened under demand and attendance stress tests;
4. any subgroup concern and its sample size;
5. an override or escalation mechanism;
6. three measures to monitor; and
7. one important limitation of the synthetic model.

Do not describe the selected policy as clinically validated or ready for autonomous deployment.

**Your memo:**

> Replace this text with your recommendation.

## Reflection questions

1. Why can average daily workload be below nominal capacity while patients still wait?
2. What information was available at scheduling time, and what was realized later?
3. Why might optimizing only utilization produce an undesirable policy?
4. How does non-preemptive urgent priority differ from interrupting a consultation?
5. What operational features are absent from this simulation—rooms, breaks, staff roles, or something else—and which omission matters most?
6. How would deployment change the future data used to retrain a no-show model?
7. What evidence would be required before applying this approach in a real clinic?

## Optional extensions

- Add a lunch break or temporary clinician absence.
- Add a room constraint separate from clinician availability.
- Compare three-clinician staffing with two-clinician overtime.
- Calculate confidence intervals across simulated days.
- Replace the supplied score with a leakage-safe model trained on synthetic historical days.
- Formulate a small constrained grid search that rejects policies exceeding an urgent-delay or overtime limit.

## Reproducibility note

The notebook creates all data in memory and writes no files. Rerunning from a fresh kernel reproduces the fixture and simulation seeds. Changing the generator or policy assumptions changes the teaching results; record any such changes in your submission.